# BirdCLEF 2025 — Exploratory Data Analysis

Understand the dataset before fine-tuning pretrained audio models.

**Data location**: `~/ps-clef2026_birdclef-0/data/birdclef-2025/`

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

DATA_DIR = Path(os.path.expanduser("~/ps-clef2026_birdclef-0/data/birdclef-2025"))
print(f"Data directory: {DATA_DIR}")
print(f"Exists: {DATA_DIR.exists()}")
if DATA_DIR.exists():
    print(f"Contents: {list(DATA_DIR.iterdir())}")

## 1. Metadata Overview

In [ ]:
# Load training metadata
meta = pd.read_csv(DATA_DIR / "train.csv")
print(f"Shape: {meta.shape}")
print(f"\nColumns: {list(meta.columns)}")
meta.head()

In [ ]:
meta.describe(include="all")

In [ ]:
meta.info()

## 2. Species Distribution

In [ ]:
# Species counts
species_col = "primary_label" if "primary_label" in meta.columns else "species"
species_counts = meta[species_col].value_counts()

print(f"Number of species: {len(species_counts)}")
print("\nTop 20 species:")
print(species_counts.head(20))
print("\nBottom 20 species:")
print(species_counts.tail(20))
print(
    f"\nRecordings per species — min: {species_counts.min()}, "
    f"median: {species_counts.median():.0f}, "
    f"max: {species_counts.max()}"
)

In [ ]:
# Species distribution plot
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Histogram of recordings per species
axes[0].hist(species_counts.values, bins=50, edgecolor="black", alpha=0.7)
axes[0].set_xlabel("Number of recordings")
axes[0].set_ylabel("Number of species")
axes[0].set_title("Distribution of recordings per species")
axes[0].axvline(
    species_counts.median(),
    color="red",
    linestyle="--",
    label=f"Median: {species_counts.median():.0f}",
)
axes[0].legend()

# Sorted species counts (log scale)
axes[1].plot(range(len(species_counts)), species_counts.values)
axes[1].set_xlabel("Species rank")
axes[1].set_ylabel("Number of recordings (log)")
axes[1].set_title("Long-tail distribution")
axes[1].set_yscale("log")

plt.tight_layout()
plt.show()

## 3. Audio Properties

In [ ]:
import librosa
import numpy as np

# Sample a few audio files to check properties
train_audio_dir = DATA_DIR / "train_audio"
if train_audio_dir.exists():
    species_dirs = sorted([d for d in train_audio_dir.iterdir() if d.is_dir()])
    print(f"Species directories: {len(species_dirs)}")

    # Sample 100 files across species
    sample_files = []
    for sp_dir in species_dirs[:50]:
        ogg_files = list(sp_dir.glob("*.ogg"))
        if ogg_files:
            sample_files.extend(ogg_files[:2])

    durations = []
    sample_rates = []
    for f in sample_files:
        try:
            info = librosa.get_duration(path=f)
            durations.append(info)
            y, sr = librosa.load(f, sr=None, duration=0.1)
            sample_rates.append(sr)
        except Exception as e:
            print(f"Error reading {f.name}: {e}")

    print(f"\nSampled {len(durations)} files")
    print(f"Sample rates: {set(sample_rates)}")
    print(
        f"Duration — min: {min(durations):.1f}s, median: {np.median(durations):.1f}s, "
        f"max: {max(durations):.1f}s, mean: {np.mean(durations):.1f}s"
    )

In [ ]:
# Duration distribution from metadata (if available)
duration_col = None
for col in ["duration", "length", "seconds"]:
    if col in meta.columns:
        duration_col = col
        break

if duration_col:
    fig, ax = plt.subplots(figsize=(10, 4))
    meta[duration_col].hist(bins=100, ax=ax, edgecolor="black", alpha=0.7)
    ax.set_xlabel("Duration (seconds)")
    ax.set_ylabel("Count")
    ax.set_title("Recording duration distribution")
    ax.axvline(5.0, color="red", linestyle="--", label="5s chunk size")
    ax.legend()
    plt.tight_layout()
    plt.show()

    print(
        f"Recordings shorter than 5s: {(meta[duration_col] < 5).sum()} ({(meta[duration_col] < 5).mean()*100:.1f}%)"
    )
    print(
        f"Recordings longer than 30s: {(meta[duration_col] > 30).sum()} ({(meta[duration_col] > 30).mean()*100:.1f}%)"
    )
else:
    print("No duration column found in metadata. Duration was sampled from audio files above.")

## 4. Geographic Coverage

In [ ]:
lat_col = "latitude" if "latitude" in meta.columns else None
lon_col = "longitude" if "longitude" in meta.columns else None

if lat_col and lon_col:
    valid_coords = meta.dropna(subset=[lat_col, lon_col])
    print(
        f"Recordings with coordinates: {len(valid_coords)} / {len(meta)} ({len(valid_coords)/len(meta)*100:.1f}%)"
    )

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.scatter(valid_coords[lon_col], valid_coords[lat_col], alpha=0.1, s=1)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_title("Geographic distribution of recordings")
    plt.tight_layout()
    plt.show()
else:
    print("No latitude/longitude columns found in metadata.")

## 5. Taxonomy

In [ ]:
taxonomy_file = DATA_DIR / "taxonomy.csv"
if taxonomy_file.exists():
    taxonomy = pd.read_csv(taxonomy_file)
    print(f"Taxonomy shape: {taxonomy.shape}")
    print(f"Columns: {list(taxonomy.columns)}")
    taxonomy.head(10)
else:
    print("No taxonomy.csv found.")

## 6. Summary

Key findings to inform model fine-tuning:

- **Number of species**: _fill after running_
- **Total recordings**: _fill after running_
- **Class imbalance**: _fill after running_
- **Audio format**: .ogg, sample rate _fill_
- **Duration range**: _fill after running_
- **Implications for training**: _fill after running_